# CT 읽기와 정상·저선량 영상 짝짓기

CT 읽기와 영상 짝짓기를 순서대로 실행하는 실습입니다.
아직 모델 학습은 하지 않습니다. 원본 CT는 읽기만 하고 결과는 `outputs/notebook`에 저장합니다.

## 실행 방법
VS Code 오른쪽 위 **커널 선택 → Python 환경 → `.venv\Scripts\python.exe`**를 선택하세요.
각 코드 셀을 위에서 아래로 **Shift+Enter**로 실행하세요. 또는 **모두 실행**을 누르세요.
변수를 바꿨다면 영향을 받는 아래 셀들도 다시 실행해야 합니다.

이 노트북은 기존 `pairs.csv` 없이도 실행됩니다. 후반부에서 영상 짝 목록을 직접 만듭니다.


## 1. 라이브러리와 프로젝트 경로
노트북에는 `__file__`이 없으므로 작업 폴더에서 프로젝트를 찾습니다. 출력된 Python 경로가 이 프로젝트의 `.venv`인지 확인하세요.

In [ ]:
from pathlib import Path
import sys
import csv
import json
import numpy as np
import pydicom
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "requirements.txt").is_file() and (p / "data/raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("ct-denoising 폴더에서 노트북을 열어 실행하세요.")
PATIENT = "L096"  # L096 등으로 바꾼 뒤 전체 셀을 다시 실행할 수 있습니다.
OUTPUT = ROOT / "outputs" / "notebook" / PATIENT
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Python:", sys.executable)
print("프로젝트:", ROOT)
print("선택한 환자:", PATIENT)

## 2. 파일 경로 찾기
`rglob`은 하위 폴더도 검색합니다. 이 단계는 파일을 읽는 것이 아니라 **경로 목록**을 만듭니다. 파일명 정렬은 구경할 첫 파일을 고르는 용도이며, 선량 영상 짝짓기에 사용하지 않습니다.

In [ ]:
patient_folder = ROOT / "data/raw/full/full_3mm" / PATIENT
files = sorted(patient_folder.rglob("*.IMA"))
if not files:
    raise FileNotFoundError(f"IMA 파일이 없습니다: {patient_folder}")
print("검색한 폴더:", patient_folder)
print("찾은 파일 개수:", len(files))
print("첫 번째 파일:", files[0])

## 3. DICOM 파일 읽기
`dcmread`는 파일을 열어 DICOM 객체 `ds`를 만듭니다. 촬영 정보와 영상 데이터가 들어 있습니다. 공개 데이터의 PatientID는 Anonymous이므로 사례는 배포 폴더 ID로 구분합니다.

In [ ]:
file_path = files[0]
ds = pydicom.dcmread(file_path)
print("객체 타입:", type(ds))
print("환자 ID:", ds.PatientID)
print("영상 종류:", ds.Modality)
print("행과 열:", ds.Rows, ds.Columns)
print("단면 두께:", ds.SliceThickness)
print("영상 위치:", ds.ImagePositionPatient)

## 4. 픽셀 배열 꺼내기
`pixel_array`가 압축 등을 해석해 NumPy 배열을 제공합니다. `(512, 512)`는 512행과 512열입니다. `[:5, :5]`는 왼쪽 위 5×5 부분입니다.

In [ ]:
pixels = ds.pixel_array
print("배열 크기:", pixels.shape)
print("자료형:", pixels.dtype)
print("최솟값 / 최댓값:", pixels.min(), pixels.max())
print("왼쪽 위 5×5 픽셀:")
print(pixels[:5, :5])

## 5. 저장된 숫자를 HU로 바꾸기
`HU = 픽셀 × RescaleSlope + RescaleIntercept`. float32로 바꾸고 배열 전체에 계산합니다. `for`문으로 픽셀을 하나씩 돌 필요가 없습니다.

In [ ]:
slope = float(ds.RescaleSlope)
intercept = float(ds.RescaleIntercept)
hu = pixels.astype(np.float32) * slope + intercept
print("Slope / Intercept:", slope, intercept)
print("HU 배열 크기:", hu.shape)
print("HU 최솟값 / 최댓값:", hu.min(), hu.max())
print("같은 픽셀 [100, 200]:", pixels[100, 200], "→", hu[100, 200])
assert np.isfinite(hu).all(), "HU 배열에 비정상 수치가 있습니다."

## 6. 첫 단면을 그림으로 보기
두께나 위치에 따라 보이는 장기가 달라집니다. 이 첫 파일은 가운데 단면이 아닙니다. HU -160~240은 **화면 표시 범위**이며 HU 배열 자체를 자르지 않습니다.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(hu, cmap="gray", vmin=-160, vmax=240)
ax.set_title(f"{PATIENT} — first full-dose slice")
ax.axis("off")
fig.colorbar(im, ax=ax, label="HU", shrink=0.7)
fig.tight_layout()
first_image = OUTPUT / "first_ct.png"
fig.savefig(first_image, dpi=120)
plt.close(fig)
display(Image(filename=str(first_image)))

## 7. 재사용할 HU 읽기 함수
앞에서 직접 수행한 읽기·변환을 함수로 묶습니다. 경로를 넣으면 HU 배열을 돌려줍니다.

In [ ]:
def read_hu(path):
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    # 저장된 픽셀 값 → Hounsfield Unit. PNG 변환 전에 이 값을 유지한다.
    return pixels * float(ds.RescaleSlope) + float(ds.RescaleIntercept)

## 8. 환자의 DICOM 헤더 모으기
`stop_before_pixels=True`는 픽셀 디코딩 없이 촬영 정보를 읽습니다. 오류 목록과 시리즈 개수도 확인합니다. 헤더 검사만으로 모든 픽셀의 디코딩 성공을 보장하지는 않습니다.

In [ ]:
def scan_patient(folder, patient):
    """픽셀은 아직 읽지 않고, 특정 환자의 헤더만 읽는다."""
    slices, errors = [], []
    for path in sorted(folder.rglob("*")):
        if not path.is_file() or patient not in path.relative_to(folder).parts:
            continue
        try:
            ds = pydicom.dcmread(path, stop_before_pixels=True)
            if ds.Modality != "CT" or int(getattr(ds, "NumberOfFrames", 1)) != 1:
                raise ValueError("단일 프레임 CT가 아닙니다")
            # 공개 데이터의 PatientID는 Anonymous이다. 사례 구분에는 폴더 ID를 쓴다.
            if str(ds.PatientID) not in (patient, "Anonymous"):
                raise ValueError(f"PatientID 불일치: {ds.PatientID}")
            slices.append({
                "path": path,
                "position": np.asarray(ds.ImagePositionPatient, dtype=float),
                "orientation": np.asarray(ds.ImageOrientationPatient, dtype=float),
                "spacing": np.asarray(ds.PixelSpacing, dtype=float),
                "shape": (int(ds.Rows), int(ds.Columns)),
                "kernel": str(ds.ConvolutionKernel),
                "thickness": float(ds.SliceThickness),
                "series": str(ds.SeriesInstanceUID),
                "dicom_patient_id": str(ds.PatientID),
            })
        except Exception as exc:
            errors.append({"path": str(path), "error": str(exc)})
    if not slices:
        raise ValueError(f"{folder}: {patient}의 유효한 CT가 없습니다. 오류: {errors[:3]}")
    if len({s["series"] for s in slices}) != 1:
        raise ValueError("시리즈가 여러 개입니다. 사용할 시리즈를 먼저 선택해야 합니다.")
    return slices, errors

In [ ]:
full, errors_full = scan_patient(ROOT / "data/raw/full", PATIENT)
quarter, errors_quarter = scan_patient(ROOT / "data/raw/quarter", PATIENT)
print("정상 / 저선량 파일 수:", len(full), len(quarter))
print("읽기 오류:", errors_full + errors_quarter)
if errors_full or errors_quarter:
    raise ValueError("읽기 오류를 해결한 뒤 진행하세요.")
print("첫 정상선량 헤더:")
display(full[0])

## 9. 같은 위치와 재구성 조건인지 비교하기
배열 크기·방향·촬영 위치·픽셀 간격·단면 두께·커널을 비교합니다. 파일명이나 익명화된 PatientID만으로 짝짓지 않습니다. 같은 배포 사례 폴더 안에서만 비교합니다.

In [ ]:
def same_geometry(a, b):
    return (
        a["dicom_patient_id"] == b["dicom_patient_id"]
        and a["shape"] == b["shape"]
        and a["kernel"] == b["kernel"]
        and np.isclose(a["thickness"], b["thickness"], atol=0.001, rtol=0)
        and np.allclose(a["position"], b["position"], atol=0.001, rtol=0)
        and np.allclose(a["orientation"], b["orientation"], atol=0.00001, rtol=0)
        and np.allclose(a["spacing"], b["spacing"], atol=0.00001, rtol=0)
    )

## 10. 영상 짝 목록 만들기
각 저선량 영상에 맞는 정상선량 영상이 정확히 하나인지 검사하고, 촬영 방향을 따라 정렬합니다. 이 정렬된 목록에서 단면 인덱스를 선택합니다.

In [ ]:
def pair_slices(full, quarter):
    pairs, used = [], set()
    for q in quarter:
        matches = [f for f in full if same_geometry(f, q)]
        if len(matches) != 1 or matches[0]["path"] in used:
            raise ValueError(f"대응하는 정상선량 영상이 유일하지 않습니다: {q['path']}")
        used.add(matches[0]["path"])
        pairs.append((matches[0], q))
    if len(used) != len(full):
        raise ValueError("짝이 없는 정상선량 슬라이스가 있습니다.")
    # 촬영 방향의 법선에 투영한 위치로 정렬한다.
    orientation = pairs[0][0]["orientation"]
    normal = np.cross(orientation[:3], orientation[3:])
    return sorted(pairs, key=lambda pair: float(np.dot(pair[0]["position"], normal)))

In [ ]:
pairs = pair_slices(full, quarter)
print("짝 개수:", len(pairs))
assert len(pairs) == len(full) == len(quarter)
index = len(pairs) // 2  # 나중에 30 등으로 바꿔 보세요.
#index = 30
f, q = pairs[index]
full_hu = read_hu(f["path"])
quarter_hu = read_hu(q["path"])
assert np.isfinite(full_hu).all() and np.isfinite(quarter_hu).all()
print("정상선량 경로:", f["path"])
print("저선량 경로:", q["path"])
print("촬영 위치:", f["position"])
print("두 배열 크기:", full_hu.shape, quarter_hu.shape)
print("평균 절대 차이 (HU):", float(np.abs(quarter_hu - full_hu).mean()))

## 11. 저선량·정상선량·차이 영상 비교
두 CT의 밝기 범위를 동일하게 표시합니다. 오른쪽은 `저선량 - 정상선량`이며 빨강은 양수, 파랑은 음수입니다. 차이 전체가 순수 노이즈의 정답인 것은 아닙니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, image, title in zip(axes[:2], [quarter_hu, full_hu],
                            ["Quarter dose", "Full dose reference"]):
    ax.imshow(image, cmap="gray", vmin=-160, vmax=240)
    ax.set_title(title + " | HU [-160, 240]")
    ax.axis("off")
diff = axes[2].imshow(quarter_hu - full_hu, cmap="coolwarm", vmin=-100, vmax=100)
axes[2].set_title("Quarter - Full (HU)")
axes[2].axis("off")
fig.colorbar(diff, ax=axes[2], shrink=0.7)
fig.suptitle(f"{PATIENT} | pair {index}/{len(pairs)} | position {f['position'].tolist()}")
fig.tight_layout()
preview_path = OUTPUT / "preview.png"
fig.savefig(preview_path, dpi=140)
plt.close(fig)
display(Image(filename=str(preview_path)))

## 12. CSV 짝 목록과 검사 결과 저장
기존 출력과 별도로 `outputs/notebook/<환자>/`에 저장합니다. CSV에는 실제 데이터 경로가 들어 있으므로 다른 PC로 옮기면 다시 생성하세요.

In [ ]:
pairs_path = OUTPUT / "pairs.csv"
with pairs_path.open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.writer(file)
    writer.writerow(["patient_id", "full_path", "quarter_path", "x_mm", "y_mm", "z_mm"])
    for full_info, quarter_info in pairs:
        writer.writerow([PATIENT, str(full_info["path"]), str(quarter_info["path"]),
                         *full_info["position"]])
report = {
    "patient": PATIENT, "full_count": len(full), "quarter_count": len(quarter),
    "paired_count": len(pairs), "shape": list(full_hu.shape), "kernel": f["kernel"],
    "thickness_mm": f["thickness"], "preview_pair_index": index,
    "dicom_patient_id": f["dicom_patient_id"], "errors": errors_full + errors_quarter,
    "note": "전체 픽셀 검사가 아니라 헤더 짝짓기 및 선택한 한 쌍의 픽셀 검사입니다.",
}
(OUTPUT / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
display(report)
print("결과 폴더:", OUTPUT)

## 13. 저장한 CSV에서 다시 읽어오기
`DictReader`는 한 행을 열 이름으로 접근 가능한 딕셔너리로 읽습니다. 파일 경로 → DICOM → HU 배열의 흐름을 다시 확인합니다.

In [ ]:
with pairs_path.open(encoding="utf-8-sig", newline="") as file:
    saved_pairs = list(csv.DictReader(file))
pair = saved_pairs[index]
full_from_csv = read_hu(pair["full_path"])
quarter_from_csv = read_hu(pair["quarter_path"])
assert np.array_equal(full_from_csv, full_hu)
assert np.array_equal(quarter_from_csv, quarter_hu)
print("CSV에서 다시 읽은 배열이 이전 배열과 같습니다.")

## 직접 해볼 작업

1. 1번 셀의 `PATIENT`를 `L096`으로 바꾸고 **모두 실행**하세요.
2. 10번의 `index`를 `30`으로 바꾸고 해당 셀부터 아래로 실행하세요.
3. `pixels`와 `hu`의 차이, `pair_slices`가 필요한 이유를 설명해 보세요.

다음 단계는 전체 환자 검사 → 환자 단위 데이터 분리 → 패치·PyTorch 텐서 만들기입니다.
아직 모델·정규화·패치 생성 코드는 포함하지 않았습니다.

출처: [AAPM/Mayo README](https://aapm.app.box.com/s/eaw4jddb53keg1bptavvvd1sf4x3pe9h/file/858370564530).


## 14. 전체 환자의 데이터 검사

In [ ]:
full_root = ROOT / "data" / "raw" / "full" / "full_3mm"
quarter_root = ROOT / "data" / "raw" / "quarter" / "quarter_3mm"

full_patients = {
    folder.name
    for folder in full_root.iterdir()
    if folder.is_dir()
}

quarter_patients = {
    folder.name
    for folder in quarter_root.iterdir()
    if folder.is_dir()
}

print("정상선량에만 있는 환자:", full_patients - quarter_patients)
print("저선량에만 있는 환자:", quarter_patients - full_patients)

assert full_patients == quarter_patients, "양쪽 환자 목록이 다릅니다."

patients = sorted(full_patients)

print("환자 수:", len(patients))
print("환자 목록:", patients)

In [ ]:
summary = []       # 환자별 검사 결과
all_pair_rows = [] # 전체 영상 짝 목록
failed_patients = []

for patient in patients:
    print(f"{patient} 검사 시작")

    try:
        full_slices, full_errors = scan_patient(
            ROOT / "data" / "raw" / "full",
            patient,
        )

        quarter_slices, quarter_errors = scan_patient(
            ROOT / "data" / "raw" / "quarter",
            patient,
        )

        header_errors = full_errors + quarter_errors

        if header_errors:
            raise ValueError(f"헤더 읽기 오류: {header_errors[:3]}")

        patient_pairs = pair_slices(full_slices, quarter_slices)

        # 헤더 검사에 더해 모든 영상의 픽셀도 실제로 읽습니다.
        for full_info, quarter_info in patient_pairs:
            for info in (full_info, quarter_info):
                image = read_hu(info["path"])

                if image.shape != info["shape"]:
                    raise ValueError(f"영상 크기 불일치: {info['path']}")

                if not np.isfinite(image).all():
                    raise ValueError(f"NaN 또는 무한대 발견: {info['path']}")

        # 환자 전체 검사가 성공한 경우에만 목록에 추가합니다.
        for full_info, quarter_info in patient_pairs:
            all_pair_rows.append({
                "patient_id": patient,
                "full_path": str(full_info["path"]),
                "quarter_path": str(quarter_info["path"]),
                "x_mm": float(full_info["position"][0]),
                "y_mm": float(full_info["position"][1]),
                "z_mm": float(full_info["position"][2]),
            })

        summary.append({
            "patient_id": patient,
            "full_count": len(full_slices),
            "quarter_count": len(quarter_slices),
            "paired_count": len(patient_pairs),
            "status": "OK",
            "error": "",
        })

        print(f"  완료: {len(patient_pairs)}쌍")

    except Exception as error:
        failed_patients.append(patient)

        summary.append({
            "patient_id": patient,
            "full_count": None,
            "quarter_count": None,
            "paired_count": None,
            "status": "FAILED",
            "error": str(error),
        })

        print(f"  실패: {error}")

In [ ]:
from IPython.display import HTML
from html import escape

columns = [
    "patient_id",
    "full_count",
    "quarter_count",
    "paired_count",
    "status",
    "error",
]

header = "".join(f"<th>{column}</th>" for column in columns)

rows = ""
for item in summary:
    cells = "".join(
        f"<td>{escape(str(item[column]))}</td>"
        for column in columns
    )
    rows += f"<tr>{cells}</tr>"

display(HTML(
    f"<table><thead><tr>{header}</tr></thead>"
    f"<tbody>{rows}</tbody></table>"
))

print("성공한 환자:", len(patients) - len(failed_patients))
print("실패한 환자:", failed_patients)
print("전체 영상 짝 수:", len(all_pair_rows))

In [ ]:
audit_folder = ROOT / "outputs" / "data_audit"
audit_folder.mkdir(parents=True, exist_ok=True)

summary_path = audit_folder / "patient_summary.csv"

with summary_path.open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.DictWriter(file, fieldnames=columns)
    writer.writeheader()
    writer.writerows(summary)

print("검사 결과 저장:", summary_path)

# 실패가 있으면 전체 학습 목록 생성은 중단합니다.
assert not failed_patients, (
    f"실패한 환자를 먼저 확인하세요: {failed_patients}"
)

manifest_path = audit_folder / "all_pairs.csv"

with manifest_path.open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "patient_id",
            "full_path",
            "quarter_path",
            "x_mm",
            "y_mm",
            "z_mm",
        ],
    )
    writer.writeheader()
    writer.writerows(all_pair_rows)

print("전체 짝 목록 저장:", manifest_path)

## 15. 환자 단위 학습·검증·테스트 분리

In [ ]:
audit_folder = ROOT / "outputs" / "data_audit"

with (audit_folder / "patient_summary.csv").open(
    encoding="utf-8-sig", newline=""
) as file:
    patient_summary = list(csv.DictReader(file))

assert len(patient_summary) == 10, "검사한 환자 수를 확인하세요."
assert all(row["status"] == "OK" for row in patient_summary), (
    "검사에 실패한 환자가 있습니다."
)

with (audit_folder / "all_pairs.csv").open(
    encoding="utf-8-sig", newline=""
) as file:
    pair_rows = list(csv.DictReader(file))

print("전체 영상 짝 수:", len(pair_rows))
print("환자 목록:", sorted({row["patient_id"] for row in pair_rows}))

In [ ]:
splits = {
    "train": ["L067", "L096", "L109", "L143", "L192", "L286", "L291"],
    "val": ["L310"],
    "test": ["L333", "L506"],
}

In [ ]:
train_ids = set(splits["train"])
val_ids = set(splits["val"])
test_ids = set(splits["test"])

# 교집합이 비어 있어야 합니다.
assert not (train_ids & val_ids), "train과 val에 같은 환자가 있습니다."
assert not (train_ids & test_ids), "train과 test에 같은 환자가 있습니다."
assert not (val_ids & test_ids), "val과 test에 같은 환자가 있습니다."

data_ids = {row["patient_id"] for row in pair_rows}
assigned_ids = train_ids | val_ids | test_ids

assert data_ids == assigned_ids, (
    f"분할에서 빠진 환자: {data_ids - assigned_ids}, "
    f"데이터에 없는 환자: {assigned_ids - data_ids}"
)

print("환자 중복 없음 / 모든 환자 배정 완료")

In [ ]:
patient_to_split = {
    patient: split_name
    for split_name, patient_ids in splits.items()
    for patient in patient_ids
}

split_rows = []

for row in pair_rows:
    new_row = row.copy()
    new_row["split"] = patient_to_split[row["patient_id"]]
    split_rows.append(new_row)

print(split_rows[0])

In [ ]:
for split_name in splits:
    selected_rows = [
        row for row in split_rows
        if row["split"] == split_name
    ]

    selected_patients = {
        row["patient_id"] for row in selected_rows
    }

    print(
        f"{split_name}: "
        f"환자 {len(selected_patients)}명 / "
        f"영상 {len(selected_rows)}쌍"
    )

assert len(split_rows) == len(pair_rows)

In [ ]:
split_folder = ROOT / "data" / "splits"
split_folder.mkdir(parents=True, exist_ok=True)

# 환자 분할 설정
(split_folder / "splits.json").write_text(
    json.dumps(splits, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# 전체 짝 목록 + split 열
fieldnames = list(pair_rows[0].keys()) + ["split"]

with (split_folder / "pairs_with_split.csv").open(
    "w", encoding="utf-8-sig", newline=""
) as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(split_rows)

# 학습·검증·테스트 목록을 별도 파일로도 저장
for split_name in splits:
    selected_rows = [
        row for row in split_rows
        if row["split"] == split_name
    ]

    with (split_folder / f"{split_name}.csv").open(
        "w", encoding="utf-8-sig", newline=""
    ) as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(selected_rows)

print("저장 위치:", split_folder)

## 16. HU 정규화와 학습용 패치 만들기

In [ ]:
train_csv = ROOT / "data" / "splits" / "train.csv"

with train_csv.open(encoding="utf-8-sig", newline="") as file:
    train_rows = list(csv.DictReader(file))

assert train_rows, "학습 목록이 비어 있습니다."
assert all(row["split"] == "train" for row in train_rows)

# 학습 목록 가운데에 있는 한 쌍 선택
row = train_rows[len(train_rows) // 2]

quarter_hu = read_hu(row["quarter_path"])
full_hu = read_hu(row["full_path"])

assert quarter_hu.shape == full_hu.shape

print("선택한 환자:", row["patient_id"])
print("영상 위치:", row["z_mm"])
print("저선량 크기:", quarter_hu.shape)
print("정상선량 크기:", full_hu.shape)

In [ ]:
HU_MIN = -1000.0
HU_MAX = 2000.0

def normalize_hu(image):
    clipped = np.clip(image, HU_MIN, HU_MAX)

    normalized = (clipped - HU_MIN) / (HU_MAX - HU_MIN)

    return normalized.astype(np.float32)

In [ ]:
quarter_normalized = normalize_hu(quarter_hu)
full_normalized = normalize_hu(full_hu)

for name, image in [
    ("저선량", quarter_normalized),
    ("정상선량", full_normalized),
]:
    print(
        name,
        "크기:", image.shape,
        "자료형:", image.dtype,
        "범위:", float(image.min()), float(image.max()),
    )

    assert np.isfinite(image).all()
    assert 0 <= image.min() <= image.max() <= 1

In [ ]:
def crop_pair(quarter, full, patch_size, rng):
    assert quarter.shape == full.shape

    height, width = quarter.shape

    if patch_size > min(height, width):
        raise ValueError("패치가 원본 영상보다 큽니다.")

    top = int(rng.integers(0, height - patch_size + 1))
    left = int(rng.integers(0, width - patch_size + 1))

    quarter_patch = quarter[
        top:top + patch_size,
        left:left + patch_size,
    ]

    full_patch = full[
        top:top + patch_size,
        left:left + patch_size,
    ]

    return quarter_patch, full_patch, top, left

In [ ]:
rng = np.random.default_rng(42)

quarter_patch, full_patch, top, left = crop_pair(
    quarter_normalized,
    full_normalized,
    patch_size=128,
    rng=rng,
)

print("자른 위치 — 행:", top, "/ 열:", left)
print("입력 패치:", quarter_patch.shape)
print("기준 패치:", full_patch.shape)

assert quarter_patch.shape == (128, 128)
assert full_patch.shape == (128, 128)

In [ ]:
from matplotlib.patches import Rectangle

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 기존 CT 표시 범위를 정규화된 숫자로 변환
display_min = (-160 - HU_MIN) / (HU_MAX - HU_MIN)
display_max = (240 - HU_MIN) / (HU_MAX - HU_MIN)

axes[0].imshow(
    quarter_normalized,
    cmap="gray",
    vmin=display_min,
    vmax=display_max,
)

axes[0].add_patch(
    Rectangle(
        (left, top),
        128,
        128,
        linewidth=2,
        edgecolor="red",
        facecolor="none",
    )
)
axes[0].set_title("Selected patch location")

axes[1].imshow(
    quarter_patch,
    cmap="gray",
    vmin=display_min,
    vmax=display_max,
)
axes[1].set_title("Input: quarter dose")

axes[2].imshow(
    full_patch,
    cmap="gray",
    vmin=display_min,
    vmax=display_max,
)
axes[2].set_title("Reference: full dose")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 17. PyTorch Dataset과 DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

print("PyTorch 버전:", torch.__version__)